# 🎬 AI Auto Video

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saigonvet8312-coder/AI_video/blob/main/AI_auto_video.ipynb)

Kịch bản → giọng đọc → hình ảnh động → video dọc 9:16.

**Trước khi chạy:** `Runtime → Change runtime type → T4 GPU`.
Chạy **Phần 1 → 2 → 3**. Từ lần sau chỉ cần chạy **Phần 2 → 3** (nếu chưa mất phiên).

## Phần 1 — Cài đặt *(một lần mỗi phiên)*

In [ ]:
%%bash
set -e
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq
apt-get install -y -qq ffmpeg fonts-noto fonts-noto-color-emoji fonts-liberation fonts-montserrat >/dev/null
pip install -q "gradio>=5,<6" omnivoice playwright numpy pillow requests anthropic deep-translator hf_transfer
python -m playwright install --with-deps chromium >/dev/null
# Font Be Vietnam Pro (thiết kế riêng cho tiếng Việt, OFL) dùng cho chữ trên hình và phụ đề; lỗi mạng thì bỏ qua
FD=/usr/local/share/fonts/be-vietnam-pro
mkdir -p "$FD"
for w in Regular Medium SemiBold Bold ExtraBold Black; do
  f="BeVietnamPro-$w.ttf"
  [ -s "$FD/$f" ] && continue
  curl -fsSL "https://github.com/google/fonts/raw/main/ofl/bevietnampro/$f" -o "$FD/$f" \
    || curl -fsSL "https://cdn.jsdelivr.net/gh/google/fonts@main/ofl/bevietnampro/$f" -o "$FD/$f" \
    || { rm -f "$FD/$f"; echo "⚠️ Không tải được font $f"; }
done
fc-cache -f >/dev/null
fc-list | grep -qi "Be Vietnam Pro" && echo "🔤 Đã cài font Be Vietnam Pro" || echo "⚠️ Chưa có Be Vietnam Pro: sẽ dùng font dự phòng"
echo "✅ Cài đặt xong"

## Phần 2 — Mã nguồn & thư mục dự án

In [ ]:
#@title Cấu hình { display-mode: "form" }
USE_DRIVE = True #@param {type:"boolean"}
VOICES_DIR = "/content/drive/MyDrive/AI_video_voices" #@param {type:"string"}
REPO_URL = "https://github.com/saigonvet8312-coder/AI_video.git" #@param {type:"string"}

import glob, os, shutil, subprocess, time
REPO_DIR = "/content/AI_video"

# Dự án chỉ lưu tạm trong phiên Colab (xong thì tải về). Chỉ thư viện giọng đọc được lưu trên Drive.
PROJECTS_DIR = "/content/projects"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    VOICES_DIR = "/content/voices"
os.makedirs(PROJECTS_DIR, exist_ok=True)
os.makedirs(VOICES_DIR, exist_ok=True)

def _norm(url):
    return url.strip().rstrip("/").removesuffix(".git").lower()

def _origin(path):
    r = subprocess.run(["git", "-C", path, "remote", "get-url", "origin"], capture_output=True, text=True)
    return r.stdout if r.returncode == 0 else ""

# Cùng repo thì cập nhật; khác repo (hoặc cập nhật lỗi) thì xoá rồi clone lại để luôn chạy đúng mã của REPO_URL
same = os.path.isdir(f"{REPO_DIR}/.git") and _norm(_origin(REPO_DIR)) == _norm(REPO_URL)
if same:
    r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True)
    if r.returncode != 0:
        print("ℹ️ Không cập nhật được, clone lại:", r.stderr.strip()[-200:])
        same = False
if not same:
    if os.path.exists(REPO_DIR):  # không xoá: chuyển thư mục cũ sang bản sao lưu để không mất dữ liệu riêng
        backup = f"{REPO_DIR}.bak-{time.strftime('%Y%m%d-%H%M%S')}"
        shutil.move(REPO_DIR, backup)
        print(f"ℹ️ Thư mục cũ được giữ lại tại {backup}")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

# Nếu repo không có thư mục app/ (các file .py nằm thẳng ở gốc), tự gom vào app/
MODULES = ["__init__", "aiimage", "aivideo", "assemble", "audiofx", "autobg", "captions", "director", "jobs", "main",
           "media", "project", "render", "scriptkit", "subjects", "templates", "tts", "vnnum", "voices"]
tracked_app = subprocess.run(["git", "-C", REPO_DIR, "ls-files", "--error-unmatch", "app/main.py"],
                             capture_output=True).returncode == 0
if not tracked_app:
    missing = [m for m in MODULES if not os.path.exists(f"{REPO_DIR}/{m}.py")]
    if not missing:
        os.makedirs(f"{REPO_DIR}/app", exist_ok=True)
        for m in MODULES:
            shutil.copy(f"{REPO_DIR}/{m}.py", f"{REPO_DIR}/app/{m}.py")
        print("ℹ️ Đã gom các file .py vào thư mục app/")
    elif len(missing) < len(MODULES):
        raise RuntimeError("Repo còn thiếu file: " + ", ".join(m + ".py" for m in missing) +
                           ". Hãy upload đủ các file rồi chạy lại.")

# Tìm thư mục chứa app/main.py (kể cả khi repo bị lồng thêm một cấp thư mục)
hits = sorted(glob.glob(f"{REPO_DIR}/**/app/main.py", recursive=True), key=len)
if not hits:
    listing = subprocess.run(["ls", "-la", REPO_DIR], capture_output=True, text=True).stdout
    raise RuntimeError(
        "Repo chưa có app/main.py. Hãy upload cả thư mục app/ lên GitHub rồi chạy lại.\n"
        f"Nội dung repo hiện tại:\n{listing}"
    )
APP_ROOT = os.path.dirname(os.path.dirname(hits[0]))
os.chdir(APP_ROOT)
os.environ["PYTHONPATH"] = APP_ROOT
os.environ["AVG_PROJECTS_DIR"] = PROJECTS_DIR
os.environ["AVG_VOICES_DIR"] = VOICES_DIR
print("✅ Mã nguồn:", APP_ROOT)
print("🎙️ Thư viện giọng:", VOICES_DIR)

# Tăng tốc tải mô hình từ Hugging Face (dùng HF_TOKEN trong Colab Secrets nếu có)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
    if _tok:
        os.environ["HF_TOKEN"] = _tok
        print("🔑 Đã dùng HF_TOKEN")
except Exception:
    print("ℹ️ Chưa có HF_TOKEN trong Secrets: tải mô hình có thể chậm.")


## Phần 3 — Mở giao diện
Chạy cell dưới, đợi vài chục giây: một **thẻ có nút "Mở giao diện"** sẽ hiện ra cùng tài khoản và mật khẩu. Bấm nút để mở web ở tab mới, nhập mật khẩu là dùng được. Cell chạy liên tục khi giao diện còn mở.

In [ ]:
#@title Phần 3 — Mở giao diện { display-mode: "form" }
SHARE_PUBLIC_LINK = True #@param {type:"boolean"}
# True (mặc định): tạo link web riêng, có mật khẩu bảo vệ. False: chỉ hiện giao diện ngay trong cell (không có link).
import os, re, subprocess, sys, threading, time, urllib.request
from IPython.display import HTML, display

env = dict(os.environ, AVG_SHARE="1" if SHARE_PUBLIC_LINK else "0", AVG_PORT="7860", PYTHONUNBUFFERED="1")
proc = subprocess.Popen([sys.executable, "-m", "app.main"], cwd=APP_ROOT, env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

_creds, _shown = {}, [False]
_NOISE = ("FutureWarning", "DeprecationWarning", "warnings.warn", "HF_HUB_ENABLE_HF_TRANSFER")

def _card(url, user, pw):
    return f"""
    <div style="font-family:system-ui,sans-serif;margin:10px 0;padding:22px 24px;border-radius:18px;max-width:640px;
         background:linear-gradient(135deg,#1b1040,#0f172a);border:1px solid #4c3a8f;color:#e8ecf8;
         box-shadow:0 12px 40px rgba(0,0,0,.45)">
      <div style="font-size:13px;letter-spacing:.14em;font-weight:800;color:#fbbf24">GIAO DIỆN ĐÃ SẴN SÀNG</div>
      <div style="font-size:24px;font-weight:800;margin:6px 0 14px">🎬 AI Auto Video</div>
      <a href="{url}" target="_blank" style="display:inline-block;padding:14px 26px;border-radius:14px;font-weight:800;font-size:18px;
         color:#0a0f1f;text-decoration:none;background:linear-gradient(135deg,#f59e0b,#ec4899 55%,#a855f7)">🚀 Mở giao diện</a>
      <div style="margin-top:16px;font-size:15px;line-height:1.7">
        Tài khoản: <b style="background:#ffffff14;padding:2px 10px;border-radius:8px">{user}</b>
        &nbsp; Mật khẩu: <b style="background:#ffffff14;padding:2px 10px;border-radius:8px">{pw}</b>
      </div>
      <div style="margin-top:10px;font-size:12.5px;color:#94a3c0">Đừng gửi link và mật khẩu cho người bạn không tin tưởng. Đóng cell này là giao diện dừng.</div>
    </div>"""

def _pump():
    for line in proc.stdout:
        m = re.search(r"tài khoản `([^`]+)` · mật khẩu `([^`]+)`", line)
        if m:
            _creds["user"], _creds["pw"] = m.group(1), m.group(2)
        u = re.search(r"https://[A-Za-z0-9-]+\.gradio\.live", line)
        if u and not _shown[0]:
            _shown[0] = True
            display(HTML(_card(u.group(0), _creds.get("user", "admin"), _creds.get("pw", "(xem dòng 🔐 ở trên)"))))
        if not any(k in line for k in _NOISE):
            print(line, end="")
threading.Thread(target=_pump, daemon=True).start()

if not SHARE_PUBLIC_LINK:
    for _ in range(180):  # chờ server sẵn sàng (tối đa ~3 phút)
        if proc.poll() is not None:
            break
        try:
            urllib.request.urlopen("http://127.0.0.1:7860", timeout=2)
            break
        except Exception:
            time.sleep(1)
    if proc.poll() is None:
        try:
            from google.colab import output
            output.serve_kernel_port_as_iframe(7860, height=950)
        except Exception as e:
            print("⚠️ Không mở được giao diện riêng tư:", e, "— hãy đặt SHARE_PUBLIC_LINK = True rồi chạy lại.")
try:
    proc.wait()
except KeyboardInterrupt:
    proc.terminate()
    print("⏹️ Đã dừng giao diện.")


## Ghi chú
- Mô hình giọng đọc tải lần đầu mất vài phút (khi bấm “Tạo giọng” lần đầu).
- **Dự án không được lưu lại** sau phiên Colab: khi video xong, hãy tải `ket-qua.zip` về ngay.
- **Giọng đọc** được lưu trên Google Drive (thư mục `AI_video_voices`): lưu một lần ở tab ②, lần sau chỉ cần chọn trong danh sách.
- Cập nhật mã: chạy lại Phần 2 rồi Phần 3. Nếu bạn đổi `REPO_URL`, thư mục mã cũ được giữ lại dưới tên `/content/AI_video.bak-…` (không bị xoá).
- API key Anthropic và token Hugging Face chỉ dùng cho từng lần bấm nút, không lưu vào dự án và không dùng chung giữa các phiên.
